In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
import tensorflow.keras.layers as layers
from tensorflow.keras.layers import Dense,Flatten,BatchNormalization, Conv2D
import tensorflow_datasets as tfds
import numpy as np
import pickle

2025-06-24 11:04:27.471473: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-24 11:04:27.471823: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-24 11:04:27.473730: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-24 11:04:27.478951: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750730667.487600   35473 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750730667.49

In [2]:
img_height = 255
img_width = 255
batch_size = 32
AUTOTUNE = tf.data.AUTOTUNE
(train_ds, val_ds, test_ds), metadata = tfds.load(
    'tf_flowers',
    split=['train[:80%]','train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
)
num_classes=metadata.features['label'].num_classes
label_name = metadata.features['label'].names
print(label_name, ", classnum : ", num_classes)

/home/hsm/python_ai/intro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 100%|██████████| 1/1 [00:07<00:00,  7.45s/ url]
2025-06-24 11:08:52.304676: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Dataset tf_flowers downloaded and prepared to /home/hsm/tensorflow_datasets/tf_flowers/3.0.1. Subsequent calls will reuse this data.
['dandelion', 'daisy', 'tulips', 'sunflowers', 'roses'] , classnum :  5


In [6]:
def prepare(ds, shuffle=False, augment=False):
    preprocess_input = tf.keras.applications.mobilenet_v3.preprocess_input
    # resize and rescale all datasets
    ds = ds.map(lambda x,y:(tf.image.resize(x,[img_height, img_width]), y), num_parallel_calls=AUTOTUNE)
    
    # 전처리적용
    ds = ds.map(lambda x,y:(preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    
    # batch all datasets
    ds = ds.batch(batch_size)
    
    #use data augmentation only on the training set
    if augment:
        data_augmentation = tf.keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.2),
        ])
        ds = ds.map(lambda x,y: (data_augmentation(x, training=True), y),num_parallel_calls=AUTOTUNE)
        
    #prefetch()
    return ds.prefetch(buffer_size=AUTOTUNE)

In [7]:
train_ds = prepare(train_ds, shuffle=True, augment=True)
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)

In [9]:
#include_top -> ANN부분 직접수정
base_model = tf.keras.applications.MobileNetV3Small(
    weights='imagenet', #load weights pre-trained on ImageNet
    input_shape = (img_height, img_width, 3), include_top = False)

# basemodel 가중치 동결
base_model.trainable=False

inputs = tf.keras.Input(shape=(img_height,img_width,3))
# 추론, 학습에서 다르게 동작하는 layer들은 추론/학습 중 하나로만 동작하게 함.
x=base_model(inputs, training=False)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
x=tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

/home/hsm/python_ai/intro/.venv/lib/python3.10/site-packages/keras/src/applications/mobilenet_v3.py:452: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


In [10]:
model.summary()
model.compile(optimizer = 'adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
history = model.fit(train_ds, epochs=15, validation_data=val_ds)
model.save('transfer_learning_flower.keras')
with open('history_flower', 'wb') as file_pi:
    pickle.dump(history.history, file_pi)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 255, 255, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 8, 8, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         2,885 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 942,005 (3.59 MB)

 Trainable params: 2,885 (11.27 KB)

 Non-trainable params: 939,120 (3.58 MB)

Epoch 1/15


2025-06-24 11:29:49.487811: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 90ms/step - accuracy: 0.4165 - loss: 1.4433 - val_accuracy: 0.8229 - val_loss: 0.5797
Epoch 2/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.7781 - loss: 0.6343 - val_accuracy: 0.8719 - val_loss: 0.4417
Epoch 3/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 79ms/step - accuracy: 0.8137 - loss: 0.5349 - val_accuracy: 0.8692 - val_loss: 0.3875
Epoch 4/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 74ms/step - accuracy: 0.8359 - loss: 0.4857 - val_accuracy: 0.8747 - val_loss: 0.3593
Epoch 5/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.8471 - loss: 0.4365 - val_accuracy: 0.8719 - val_loss: 0.3416
Epoch 6/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.8595 - loss: 0.4106 - val_accuracy: 0.8910 - val_loss: 0.3310
Epoch 7/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 79ms/step - accuracy: 0.8631 - loss: 0.3766 - val_accuracy: 0.8883 - val_loss: 0.3241
Epoch 8/15
92/92 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - accuracy: 0.8658 - loss: 0.3927 - val_accuracy: 0.8910 - val_loss: 0